# 02. Graph Construction

**Input** : `Preprocessing_0512.csv` (44,776 nodes × 36 cols)

**Output**: 7가지 엣지 조합 graph 파일

## 엣지 종류
| 엣지 | 연결 기준 | 근거 |
|---|---|---|
| R-U-R | 같은 유저(user_int) | 유저 행동 패턴 공유 |
| R-T-R | 같은 식당 + 같은 연월 | 조직적 캠페인 패턴 |
| R-X-R | 같은 식당 + 극단 별점(1/5) | 별점 조작 포착 |
| R-S-R | 같은 식당 + 같은 별점(1~5) | 동일 평점 그룹 패턴 |

## 실험 조합
| 조합 | 파일명 |
|---|---|
| R-U-R | `graph_RUR.pt` |
| R-T-R | `graph_RTR.pt` |
| R-X-R | `graph_RXR.pt` |
| R-S-R | `graph_RSR.pt` |
| R-U-R + R-T-R | `graph_RUR_RTR.pt` |
| R-U-R + R-X-R | `graph_RUR_RXR.pt` |
| R-T-R + R-X-R | `graph_RTR_RXR.pt` |
| R-U-R + R-T-R + R-X-R | `graph_RUR_RTR_RXR.pt` |
| **R-U-R + R-S-R** | **`graph_RUR_RSR.pt`** |
| R-U-R + R-T-R + R-S-R | `graph_RUR_RTR_RSR.pt` |

In [1]:
import pandas as pd
import numpy as np
import torch
from torch_geometric.data import Data
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from itertools import combinations
import os

BASE_DIR  = os.path.abspath(".")
DATA_PATH = os.path.join(BASE_DIR, "Preprocessing_0512.csv")
GRAPH_DIR = os.path.join(BASE_DIR, "graphs")
os.makedirs(GRAPH_DIR, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {device}")
if device.type == "cuda":
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
print(f"Data   : {DATA_PATH}")
print(f"GraphDir: {GRAPH_DIR}")

c:\Users\user\anaconda3\envs\gnn\Lib\site-packages\torch_geometric\__init__.py:4: UserWarning: An issue occurred while importing 'torch-scatter'. Disabling its usage. Stacktrace: Could not load this library: C:\Users\user\anaconda3\envs\gnn\Lib\site-packages\torch_scatter\_version_cuda.pyd
  import torch_geometric.typing
c:\Users\user\anaconda3\envs\gnn\Lib\site-packages\torch_geometric\__init__.py:4: UserWarning: An issue occurred while importing 'torch-sparse'. Disabling its usage. Stacktrace: Could not load this library: C:\Users\user\anaconda3\envs\gnn\Lib\site-packages\torch_sparse\_version_cuda.pyd
  import torch_geometric.typing


Device : cuda
GPU    : NVIDIA GeForce RTX 5070 Ti Laptop GPU
Data   : c:\Users\user\Desktop\project_DScover_연합학술제_GNN\pipeline_1\Preprocessing_0512.csv
GraphDir: c:\Users\user\Desktop\project_DScover_연합학술제_GNN\pipeline_1\graphs


## Step 1. 데이터 로드

In [2]:
df = pd.read_csv(DATA_PATH)

print(f"Shape      : {df.shape}")
print(f"노드 수     : {len(df):,}")
print(f"사기 비율   : {df['label'].mean()*100:.2f}%")
print(f"고유 유저    : {df['user_int'].nunique():,}")
print(f"고유 식당    : {df['prod_int'].nunique():,}")
print(f"연월 목록    : {sorted(df['year_month'].unique())}")

Shape      : (44776, 36)
노드 수     : 44,776
사기 비율   : 14.64%
고유 유저    : 28,585
고유 식당    : 3,214
연월 목록    : ['2013-03', '2013-04', '2013-07', '2013-08']


## Step 2. 노드 피처 정규화 (24개)

In [3]:
feat_cols = [
    # A. Sentiment (4개) — emo_pos/emo_neg 유지 (변별력 높음)
    "emo_pos", "emo_neg", "emotion", "emo_anger",
    # B. Text Authenticity (6개)
    "WC", "avg_word_len", "caps_ratio", "contraction_ratio", "oov_ratio", "punct_ratio",
    # C. Template Proxy (2개) — bigram_repeat_ratio 제거
    "type_token_ratio", "Analytic",
    # D. POS & Pronoun (5개)
    "verb", "adj", "adverb", "ppron_i", "focuspast",
    # E. Behavioral (4개)
    "user_review_cnt", "user_inter_review_days", "text_rating_mismatch", "review_burstiness",
    # F. Network Proxy (3개) — user_prod_diversity 제거
    "prod_rating_entropy", "user_extreme_ratio", "rating_deviation"
]  # 총 24개

scaler = StandardScaler()
X = scaler.fit_transform(df[feat_cols].values).astype(np.float32)
y = df["label"].values.astype(np.int64)

x_tensor = torch.tensor(X, dtype=torch.float)
y_tensor = torch.tensor(y, dtype=torch.long)

print(f"노드 피처 shape : {x_tensor.shape}  (nodes × {len(feat_cols)})")
print(f"라벨 분포       : 정상={(y==0).sum():,}  사기={(y==1).sum():,}")

# Train/Val/Test 마스크 (고정)
idx = np.arange(len(df))
idx_train, idx_test = train_test_split(idx, test_size=0.2, random_state=42, stratify=y)
idx_train, idx_val  = train_test_split(idx_train, test_size=0.1, random_state=42, stratify=y[idx_train])

train_mask = torch.zeros(len(df), dtype=torch.bool)
val_mask   = torch.zeros(len(df), dtype=torch.bool)
test_mask  = torch.zeros(len(df), dtype=torch.bool)
train_mask[idx_train] = True
val_mask[idx_val]     = True
test_mask[idx_test]   = True
print(f"Train: {train_mask.sum():,}  Val: {val_mask.sum():,}  Test: {test_mask.sum():,}")

노드 피처 shape : torch.Size([44776, 24])  (nodes × 24)
라벨 분포       : 정상=38,221  사기=6,555
Train: 32,238  Val: 3,582  Test: 8,956


## Step 3. 엣지 생성 (4종 개별 계산)

In [4]:
# ── R-U-R: 같은 유저 ─────────────────────────────────
print("R-U-R 생성 중...")
rur_edges = []
for uid, group in df.groupby("user_int"):
    nodes = group["node_id"].tolist()
    if len(nodes) > 1:
        for a, b in combinations(nodes, 2):
            rur_edges += [(a,b),(b,a)]
rur_arr = np.array(rur_edges, dtype=np.int64) if rur_edges else np.empty((0,2), dtype=np.int64)
print(f"  R-U-R 엣지 수 : {len(rur_arr):,}")

# ── R-T-R: 같은 식당 + 같은 연월 ─────────────────────
print("R-T-R 생성 중...")
rtr_edges = []
for (pid, ym), group in df.groupby(["prod_int", "year_month"]):
    nodes = group["node_id"].tolist()
    if len(nodes) > 1:
        for a, b in combinations(nodes, 2):
            rtr_edges += [(a,b),(b,a)]
rtr_arr = np.array(rtr_edges, dtype=np.int64) if rtr_edges else np.empty((0,2), dtype=np.int64)
print(f"  R-T-R 엣지 수 : {len(rtr_arr):,}")

# ── R-X-R: 같은 식당 + 극단 별점(1/5) ────────────────
print("R-X-R 생성 중...")
rxr_edges = []
extreme_df = df[df["rating"].isin([1, 5])]
for pid, group in extreme_df.groupby("prod_int"):
    nodes = group["node_id"].tolist()
    if len(nodes) > 1:
        for a, b in combinations(nodes, 2):
            rxr_edges += [(a,b),(b,a)]
rxr_arr = np.array(rxr_edges, dtype=np.int64) if rxr_edges else np.empty((0,2), dtype=np.int64)
print(f"  R-X-R 엣지 수 : {len(rxr_arr):,}")

print(f"\n총 극단 별점 리뷰: {len(extreme_df):,}개 ({len(extreme_df)/len(df)*100:.1f}%)")

# ── R-S-R: 같은 식당 + 같은 별점(1~5) ───────────────
print("R-S-R 생성 중...")
rsr_edges = []
for (pid, rating), group in df.groupby(["prod_int", "rating"]):
    nodes = group["node_id"].tolist()
    if len(nodes) > 1:
        for a, b in combinations(nodes, 2):
            rsr_edges += [(a,b),(b,a)]
rsr_arr = np.array(rsr_edges, dtype=np.int64) if rsr_edges else np.empty((0,2), dtype=np.int64)
print(f"  R-S-R 엣지 수근: {len(rsr_arr):,}")

# 별점 분포 확인
print("\n별점 분포:")
for r, cnt in df['rating'].value_counts().sort_index().items():
    print(f"  별점 {r}: {cnt:,}개 ({cnt/len(df)*100:.1f}%)")

R-U-R 생성 중...
  R-U-R 엣지 수 : 89,504
R-T-R 생성 중...
  R-T-R 엣지 수 : 733,708
R-X-R 생성 중...
  R-X-R 엣지 수 : 542,348

총 극단 별점 리뷰: 19,497개 (43.5%)
R-S-R 생성 중...
  R-S-R 엣지 수근: 890,856

별점 분포:
  별점 1: 2,811개 (6.3%)
  별점 2: 3,204개 (7.2%)
  별점 3: 6,132개 (13.7%)
  별점 4: 15,943개 (35.6%)
  별점 5: 16,686개 (37.3%)


## Step 4. 7가지 엣지 조합으로 graph 저장

In [5]:
EDGE_COMBINATIONS = [
    ("RUR",             [rur_arr]),
    ("RTR",             [rtr_arr]),
    ("RXR",             [rxr_arr]),
    ("RUR_RTR",         [rur_arr, rtr_arr]),
    ("RUR_RXR",         [rur_arr, rxr_arr]),
    ("RTR_RXR",         [rtr_arr, rxr_arr]),
    ("RUR_RTR_RXR",     [rur_arr, rtr_arr, rxr_arr]),
    ("RSR",             [rsr_arr]),
    ("RUR_RSR",         [rur_arr, rsr_arr]),
    ("RUR_RTR_RSR",     [rur_arr, rtr_arr, rsr_arr]),
]

print(f"{'조합':<20} {'엣지 수':>12}  {'관계':>4}  {'저장 파일'}")
print("-" * 65)

for name, edge_arrays in EDGE_COMBINATIONS:
    # 관계 타입별로 독립 dedup → edge_type 생성 (RGCN용)
    all_edges, all_types = [], []
    for type_idx, arr in enumerate(edge_arrays):
        unique_arr = np.unique(arr, axis=0)
        all_edges.append(unique_arr)
        all_types.append(np.full(len(unique_arr), type_idx, dtype=np.int64))

    combined_edges = np.concatenate(all_edges, axis=0)
    combined_types = np.concatenate(all_types, axis=0)
    num_rels       = len(edge_arrays)

    edge_index    = torch.tensor(combined_edges.T, dtype=torch.long)
    edge_type_t   = torch.tensor(combined_types,   dtype=torch.long)

    data = Data(
        x            = x_tensor,
        edge_index   = edge_index,
        y            = y_tensor,
        train_mask   = train_mask,
        val_mask     = val_mask,
        test_mask    = test_mask,
        edge_type    = edge_type_t,
        num_relations = num_rels,
    )

    save_path = os.path.join(GRAPH_DIR, f"graph_{name}.pt")
    torch.save(data, save_path)
    print(f"{name:<20} {edge_index.shape[1]:>12,}  {num_rels:>4}  graph_{name}.pt")

print("-" * 65)
print(f"\n저장 완료: {GRAPH_DIR}")
print(f"총 {len(EDGE_COMBINATIONS)}개 그래프 파일 생성 (edge_type 포함)")

조합                           엣지 수  저장 파일
------------------------------------------------------------
RUR                        89,504  graph_RUR.pt
RTR                       733,708  graph_RTR.pt
RXR                       542,348  graph_RXR.pt
RUR_RTR                   823,212  graph_RUR_RTR.pt
RUR_RXR                   631,852  graph_RUR_RXR.pt
RTR_RXR                 1,126,604  graph_RTR_RXR.pt
RUR_RTR_RXR             1,216,108  graph_RUR_RTR_RXR.pt
RSR                       890,856  graph_RSR.pt
RUR_RSR                   980,360  graph_RUR_RSR.pt
RUR_RTR_RSR             1,469,218  graph_RUR_RTR_RSR.pt
------------------------------------------------------------

저장 완료: c:\Users\user\Desktop\project_DScover_연합학술제_GNN\pipeline_1\graphs
총 10개 그래프 파일 생성
